In [1]:
%%html

<h1>Initializing Environment</h1>

In [230]:
%%capture 

import pandas as pd
import numpy as np
from transformers import RobertaModel, RobertaTokenizer
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import os
from itertools import cycle


df = pd.read_csv("allsides-balanced.csv")

model = RobertaModel.from_pretrained("roberta-base")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

In [198]:
%%html

<h1>Cleaning & Understanding Data</h1>

<ul>
    <li>Removing index column</li>
    <li>Checking shape after each transformation</li>
    <li>Removing null and duplicate values</li>
    <li>Reset indices due to deleted rows</li>
</ul>

In [232]:
df = df.drop(columns=df.columns[0])


print(f"Initial Data Shape: {df.shape}\n\nData Types: \n{df.dtypes}\n")
print(f"NA Values:\n{df.isna().sum()}\n\nNull Values:\n{df.isnull().sum()}\n\n")


df = df.dropna(subset=["source", "text"])
print(f"After Removing NA: {df.shape}\n")


df = df.drop_duplicates(subset=["text"])
print(f"After Removing Duplicate Articles: {df.shape}\n")


df = df.reset_index(drop=True)

Initial Data Shape: (21754, 6)

Data Types: 
title          str
tags           str
heading        str
source         str
text           str
bias_rating    str
dtype: object

NA Values:
title          0
tags           0
heading        0
source         8
text           7
bias_rating    0
dtype: int64

Null Values:
title          0
tags           0
heading        0
source         8
text           7
bias_rating    0
dtype: int64


After Removing NA: (21739, 6)

After Removing Duplicate Articles: (21715, 6)



In [234]:
%%html

<h1>Feature Extraction Pipeline (roBERTa)</h1>

<p>
Data extraction pipeline to tokenize and convert article features into 768d float embeddings using batch processing of 32 articles at a time.
Mean-pooling utilized to calculate a culumative representational embedding for each article
</p>

In [236]:
def feature_extraction(batch_size = 32):

    features = []
    
    # Utilizing GPU for data processing (M1)
    device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
    model.to(device)
    model.eval()
    
    article_text = df["text"].tolist()
    
    for i in range(0, len(article_text), batch_size):
    
        batch = article_text[i : i + batch_size]
    
        input_articles = tokenizer(
            batch,
            padding = True,
            truncation = True,
            max_length = 512,
            return_tensors = "pt"
        )
    
        # Convert tensor outputs (CPU -> GPU)
        input_articles = { k: v.to(device) for k, v in input_articles.items() }
    
        with torch.no_grad():
            output_articles = model(**input_articles)

            # --- Mean Pooling 0---
    
            # Increase dimension of attention_mask to output_articles for to enable multiplication
            attention_mask = input_articles["attention_mask"].unsqueeze(-1)
    

            # Masked embeddings multiplication will zero out padding tokens 
            masked_embeddings = output_articles.last_hidden_state * attention_mask
       
    
            # Sum real tokens embedding, and divide by number of tokens per article
            mean_batch = masked_embeddings.sum(1) / attention_mask.sum(1)
            

            # Append features
            features.append(mean_batch.cpu())    


            # --- CLS Extraction ---

            # cls_batch = output_articles.last_hidden_state[:, 0, :]

            # features.append(cls_batch.cpu())

    return features

features = feature_extraction()
article_features = torch.cat(features, dim=0)

In [165]:
%%html

<h1>Defining Data Splits</h1>

<h3>What are our data-splits we are testing on?</h3>

<p>
Forming various data splits to test DANN on with roBERTa positional embeddings. We are testing various splits due to a key limitation identified
in the IQBias all-sides-balanced.csv dataset, that only has one political leaning per source outlet. For example, Fox News only contains articles
that have the right-leaning bias label, without containing article information that maps to left or center leaning.
</p>

<p>

<h5>(1) Grouping articles by triplets (common title maps -> (left, center, right) article)</h5>
    
<ul>
    <li>Group triplets, remove non-triplets</li>
    <li>Source is considered the article "title"</li>
</ul>
 
<h5>(2) Group outlets by outlet source (at training time) and testing on unseen outlets (at test time)</h5>

<ul>
    <li>May include class imbalance between (left, center, right) articles</li>
    <li>May not generalize during testing since each outlet has a distinct writing style during training</li>
</ul>
           
<h5>(3) Separate domain as source non-Fox outlet, and target Fox outlet articles</h5>

<ul>
    <li>May be limited since Fox news only contains right leaning articles</li>
    <li>May cause leakage, improper generalization</li>
</ul>

</p>

In [238]:
# -------- Split (1) --------

df_triplet = df.copy()

df_triplet["title"] = df_triplet["title"].str.strip().str.lower()

is_triplet = df_triplet.groupby("title")["bias_rating"].apply(
    lambda x: sorted(x.tolist()) == ["center", "left", "right"]
)

df_triplet = df_triplet[df_triplet["title"].isin(is_triplet[is_triplet].index)]

print(f"Triplet Bias Distribution:\n\n{df_triplet["bias_rating"].value_counts()}\n\nTotal Sources:\n\n{len(df_triplet)}")


# -------- Split (2) --------


df_unseen = df.copy()

print(f"Unseen Seen Bias Distribution:\n\n{df_unseen["bias_rating"].value_counts()}\n\nTotal Sources:\n\n{len(df_unseen)}")


seen = [
    # Left-leaning
    "CNN (Online News)", "Washington Post", "HuffPost", "NPR (Online News)", "Associated Press",
    "Vox", "The Guardian", "ABC News (Online)", "Bloomberg",
    
    # Center
    "The Hill", "Reuters", "Axios", "CNBC",  "Newsweek", "Christian Science Monitor",
    
    # Right-leaning
    "Washington Times", "Washington Examiner", "National Review", "The Blaze", "Breitbart News", 
    "CBN", "The Epoch Times", "Reason",
]

unseen = [
    # Left-leaning
    "New York Times (News)", "Politico", "NBC News (Online)",
    
    # Center
    "Wall Street Journal (News)", "BBC News",
    
    # Right-leaning
    "Fox News (Online News)", "Townhall", "Newsmax (News)",
]


# -------- Split (3) --------

df_fox = df.copy()

fox_outlets = ["Fox News (Online News)", "Fox Business", "Fox News (Opinion)", "Fox News Latino"]

Triplet Bias Distribution:

bias_rating
left      3965
center    3965
right     3965
Name: count, dtype: int64

Total Sources:

11895
Unseen Seen Bias Distribution:

bias_rating
left      10260
right      7214
center     4241
Name: count, dtype: int64

Total Sources:

21715


In [7]:
def save_split(split_type, to_dir):

    bias_rating_map = {"left" : 0, "center": 1, "right": 2}
    
    match split_type:

        case "triplet":

            triplet_features = article_features[df_triplet.index].cpu()
            triplet_labels = torch.tensor(df_triplet["bias_rating"].map(bias_rating_map).tolist())

            # Randomize order of titles
            unique_titles = np.random.permutation(df_triplet["title"].unique())

            # Create masks
            source_mask = torch.tensor([t in set(unique_titles[:int(0.8 * len(unique_titles))]) for t in df_triplet["title"].tolist()])
            target_mask = ~source_mask
            
            # Split embeddings
            train_embedding, train_labels = triplet_features[source_mask].cpu(), triplet_labels[source_mask]
            target_embedding, target_labels = triplet_features[target_mask].cpu(), triplet_labels[target_mask]


        case "unseen":

            unseen_labels = torch.tensor(df_unseen["bias_rating"].map(bias_rating_map).tolist())

             # Create masks
            source_mask = torch.tensor([s in seen for s in df_unseen["source"].tolist()])
            target_mask = torch.tensor([s in unseen for s in df_unseen["source"].tolist()])

            df_source = df_unseen[source_mask.numpy().astype(bool)]
            df_target = df_unseen[target_mask.numpy().astype(bool)]

            print(df_source["bias_rating"].value_counts())
            print(df_target["bias_rating"].value_counts())

            # Split embeddings
            train_embedding, train_labels = article_features[source_mask].cpu(), unseen_labels[source_mask]
            target_embedding, target_labels = article_features[target_mask].cpu(), unseen_labels[target_mask]
            
        case "fox":
            
            fox_labels  = torch.tensor(df_fox["bias_rating"].map(bias_rating_map).tolist())
            
            # Create masks
            source_mask = torch.tensor([s not in fox_outlets for s in df_fox["source"].tolist()])
            target_mask = ~source_mask

            # Split embeddings
            train_embedding, train_labels = article_features[source_mask].cpu(), fox_labels[source_mask]
            target_embedding, target_labels = article_features[target_mask].cpu(), fox_labels[target_mask]

    if os.path.isdir(to_dir): print(f"{to_dir} directory already exists!") 
    
    else:
        os.mkdir(to_dir)

        torch.save(train_embedding, f"{to_dir}/train_embedding.pt")
        torch.save(train_labels, f"{to_dir}/train_labels.pt")
        torch.save(target_embedding, f"{to_dir}/test_embedding.pt")
        torch.save(target_labels, f"{to_dir}/test_labels.pt")


save_split("triplet", "triplet_dir")
save_split("unseen", "unseen_dir")
save_split("fox", "fox_dir")

def load_split(from_dir):

    source_embedding = torch.load(f"{from_dir}/train_embedding.pt").float()
    source_labels = torch.load(f"{from_dir}/train_labels.pt")
    target_embedding = torch.load(f"{from_dir}/test_embedding.pt").float()
    target_labels = torch.load(f"{from_dir}/test_labels.pt")
    
    # Source train (labels for Gy, Gd)
    source_loader = DataLoader(
        TensorDataset(source_embedding, source_labels),
        batch_size=32, 
        shuffle=True
    )
    
    # Target train (no labels, Gd only)
    target_loader = DataLoader(
        TensorDataset(target_embedding),
        batch_size=32, 
        shuffle=True
    )

    return source_loader, target_loader, target_embedding, target_labels

NameError: name 'article_features' is not defined

In [242]:
%%html

<h1>DANN Implementation</h1>

<h3>
Below is a quick reference table to the different components of the Domain Adversarial Neural Network implementation adapted from
Ganin et al. (<a href="https://arxiv.org/abs/1505.07818">Academic Paper</a>)
</h3>


<h5>(1) Gradient Reversal Layer</h5>
<h5>(2) Feature Extractor</h5>
<h5>(3) Adversarial Classifier</h5>
<h5>(4) Label Classifier</h5>
<h5>(5) DANN</h5>

In [26]:
# -------- (1) --------

class ReversalFunction(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, lam):

        ctx.save_for_backward(torch.tensor(lam))
        
        return x.clone()

    @staticmethod
    def backward(ctx, gradient):

        lam, = ctx.saved_tensors
        
        return -lam * gradient, None

class GradientReversalLayer(nn.Module):

    def __init__(self):

        super().__init__()
        self.lam = 0.0

    def forward(self, x):
        return ReversalFunction.apply(x, self.lam)


# -------- (2) --------

class FeatureExtractor(nn.Module):

    def __init__(self):
        super().__init__()

        # Simple Feed-forward Network (Compress embeddings into domain-invariant features)
        self.network = nn.Sequential(
 
            nn.Linear(768, 256), # Reduce roBERTa embedding from 768 -> 256 representation
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128), # Reduce roBERTa embedding from 256 -> 128 representation
            nn.ReLU(),
            nn.Dropout(0.3)
        
        )

    def forward(self, x):
        return self.network(x)


# -------- (3) --------

class AdversarialClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.grl = GradientReversalLayer()
        self.network = nn.Sequential(
            
            nn.Linear(128, 64), # Compressing 128 representation to 64
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1) # Compressing 64 representation to 1 binary output
        )

    def forward(self, input_features):
        return self.network(self.grl(input_features))


# -------- (4) --------

class LabelClassifier(nn.Module):

    def __init__(self):
        super().__init__()
        
        self.network = nn.Linear(128, 3) # Compressing 128 representation to 3 output(s) (Left, Center, or Right)

    def forward(self, input_features):
        return self.network(input_features)


# -------- (5) --------

class DANN(nn.Module):
    def __init__(self):
        super().__init__()

        self.feature_extractor = FeatureExtractor()
        self.label_classifier = LabelClassifier()
        self.adversarial_classifier = AdversarialClassifier()

    def forward(self, x):

        features = self.feature_extractor(x)

        return self.label_classifier(features), self.adversarial_classifier(features)

In [42]:
%%html

<h1>Testing & Results</h1>

<p>
(Still in progress ...)
</p>

In [38]:
def train(from_dir, epochs=30, lr=1e-3, y=10, no_grl=True):
    
    source_loader, target_loader, target_embedding, target_labels = load_split(from_dir)

    # Enable M1 GPU
    device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
    model = DANN().to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Label Classification (Left, Right, Center)
    label_loss = nn.CrossEntropyLoss()

    # Domain Classifier (Source, Target)
    adversarial_loss = nn.BCEWithLogitsLoss()
    
    # Track steps
    steps = epochs * len(source_loader)
    current_step = 0
    
    for epoch in range(epochs):
        
        model.train()

        # Prevent target loader exhaustion
        target = cycle(target_loader)
        
        epoch_label_loss, epoch_domain_loss = 0.0, 0.0
        
        for (source_x, source_y) in source_loader:
            
            # Lambda Scheduling (lambda bounded from -> (0,1))
            p = current_step / steps
            lam = 2 / (1 + np.exp(-y * p)) - 1

            # Without DANN
            if no_grl:
                model.adversarial_classifier.grl.lam = 0

            # With DANN
            else:
                model.adversarial_classifier.grl.lam = lam
            
            # Unpack next target batch
            (target_x,) = next(target)
            
            source_x = source_x.to(device)
            source_y = source_y.to(device)
            target_x = target_x.to(device)
            
            # Domain labels (source, target)
            source_domain = torch.zeros(source_x.size(0), 1).to(device)
            target_domain = torch.ones(target_x.size(0), 1).to(device)
            
            # On foward pass, generate source and domain prediction 
            source_label_pred, source_domain_pred = model(source_x)
            
            # Target labels unknown
            _, target_domain_pred = model(target_x)
            
            # Compute source loss
            loss_label = label_loss(source_label_pred, source_y)
            
            # Calculate domain loss (source and target)
            loss_domain = adversarial_loss(source_domain_pred, source_domain) + adversarial_loss(target_domain_pred, target_domain)
            
            loss = loss_label + loss_domain
            
            # Back-propagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_label_loss += loss_label.item()
            epoch_domain_loss += loss_domain.item()
            current_step += 1
        
        # Average label & domain loss
        avg_ll, avg_dl = epoch_label_loss / len(source_loader), epoch_domain_loss / len(source_loader)

        # Training summary (epoch, lamda, label & domain loss)
        print(f"[Epoch: {epoch+1}/{epochs} - Lambda: {lam:.5f} - Label Loss: {avg_ll:.5f} - Domain Loss: {avg_dl:.5f}]")
    
    return model, target_embedding, target_labels

In [ ]:
%%capture

triplet_model_no_lam, triplet_target_embedding_no_lam, triplet_target_labels_no_lam = train("triplet_dir")
triplet_model, triplet_target_embedding, triplet_target_labels = train("triplet_dir", no_grl=False)
unseen_model, unseen_target_embedding, unseen_target_labels = train("unseen_dir")
fox_model, fox_target_embedding, fox_target_labels = train("fox_dir")

[Epoch: 1/30 - Lambda: 0.16460 - Label Loss: 1.09774 - Domain Loss: 1.38652]
[Epoch: 2/30 - Lambda: 0.32101 - Label Loss: 1.07725 - Domain Loss: 1.38653]
[Epoch: 3/30 - Lambda: 0.46168 - Label Loss: 1.04934 - Domain Loss: 1.38611]
[Epoch: 4/30 - Lambda: 0.58241 - Label Loss: 1.03117 - Domain Loss: 1.38627]
[Epoch: 5/30 - Lambda: 0.68196 - Label Loss: 1.02077 - Domain Loss: 1.38533]
[Epoch: 6/30 - Lambda: 0.76136 - Label Loss: 1.01076 - Domain Loss: 1.38595]
[Epoch: 7/30 - Lambda: 0.82302 - Label Loss: 1.00110 - Domain Loss: 1.38576]
[Epoch: 8/30 - Lambda: 0.86993 - Label Loss: 0.99969 - Domain Loss: 1.38564]
[Epoch: 9/30 - Lambda: 0.90505 - Label Loss: 0.99139 - Domain Loss: 1.38565]
[Epoch: 10/30 - Lambda: 0.93104 - Label Loss: 0.98905 - Domain Loss: 1.38538]
[Epoch: 11/30 - Lambda: 0.95010 - Label Loss: 0.98682 - Domain Loss: 1.38503]
[Epoch: 12/30 - Lambda: 0.96399 - Label Loss: 0.97697 - Domain Loss: 1.38590]
[Epoch: 13/30 - Lambda: 0.97406 - Label Loss: 0.98058 - Domain Loss: 1.38

KeyboardInterrupt: 